# Teste de mesclagem com Extra Trees no lugar do Ridge

Este notebook repete o teste de mesclagem por media ponderada, trocando o terceiro algoritmo: sai o
**Ridge**, entra o **Extra Trees**. Os outros dois continuam sendo Random Forest e Gradient Boosting.

## Por que trocar

No teste anterior, o Ridge zerou. Em todas as sete areas o peso dele na melhor combinacao foi zero,
e sozinho ele errava de longe mais que os outros dois - 19,8 graus no coil_temps, contra 11,4 do
Random Forest. Um modelo linear nao tem como competir com arvores nesta base, e cada decimo de peso
dado a ele era peso tirado de quem erra menos.

O Extra Trees e um candidato melhor por dois motivos. O primeiro e que ele joga no mesmo nivel: e um
conjunto de arvores, como os outros dois, e o projeto ja o mediu no
`treino_extratrees_7areas.ipynb`. O segundo e mais interessante para uma mescla: ele **sorteia** os
pontos de corte de cada divisao, em vez de procurar o melhor. Isso o torna individualmente um pouco
pior, mas faz com que ele erre de forma **diferente** do Random Forest - e e justamente da diferenca
entre os erros que uma mescla tira proveito. Um modelo que erra igual a outro nao acrescenta nada
quando os dois sao promediados.

## A pergunta deste notebook

Duas, na verdade, e a segunda e a que importa:

1. Qual a melhor combinacao de pesos de cada area, agora com Extra Trees na mescla?
2. **O Extra Trees ganha o seu lugar?** Ou seja: a melhor combinacao que usa Extra Trees erra menos
   do que a melhor combinacao que nao usa? Se nao erra, ele nao acrescenta nada, e o resultado do
   teste anterior continua valendo.

A Secao 9 responde a segunda de forma direta, comparando as duas dentro da mesma grade.

## O que este teste nao faz

Como o anterior: nao toca no conjunto de teste reservado, nao gera arquivo de dados novo e nao
altera nenhum notebook existente. Todas as metricas vem da validacao cruzada sobre os ensaios de
treino.

## 1. Configuracao

Este notebook roda a bancada inteira **duas vezes**, sobre as duas versoes de tratamento das perdas:

- `dataset_<area>_13cat_snull.csv` - as perdas em **13 categorias**, uma coluna por combinacao
  declarada no motivo do ensaio.
- `dataset_<area>_perfil_snull.csv` - as mesmas perdas **agrupadas por perfil**, o que reduz as 13
  colunas a tres (`losses_miro`, `losses_simone`, `losses_no losses`).

O sufixo `_snull` e obrigatorio nas duas: sao as versoes com os nulos das colunas-alvo ja tratados na
preparacao. Usar uma base com nulos e a outra sem tornaria a comparacao entre as duas sem sentido,
porque as duas estariam medindo sobre conjuntos de observacoes diferentes.

A terceira peca e `particao_treino_teste.csv`, a particao fixa de treino e teste, com a coluna
`dobra` que define os grupos da validacao cruzada. Ela e a mesma para as duas bases - e o que garante
que a diferenca medida entre elas venha da codificacao das perdas, e nao de um sorteio diferente.

A base de **referencia** e a `_13cat_snull`: e sobre ela que correm as secoes 3 a 10, que tratam da
mescla. A comparacao entre as duas bases esta na Secao 11, no fim.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/testes_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os, warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

# As duas versoes de tratamento das perdas que este notebook compara. Ambas na
# variante _snull, com os nulos das colunas-alvo ja tratados na preparacao.
SUFIXOS = ["_13cat_snull", "_perfil_snull"]

# Nome curto de cada base, para as tabelas.
NOME_BASE = {"_13cat_snull": "13 categorias", "_perfil_snull": "perfil de perdas"}

# Base de referencia: a que alimenta as secoes sobre a mescla. A outra entra na
# comparacao final.
SUFIXO = SUFIXOS[0]

AREAS = ["coil_temps", "ct_knobs", "ipc_comp", "ipc_cooling",
         "main_vent_sys", "mainback", "oven_hmi"]

# Pastas candidatas, da mais especifica para a mais geral. A particao nao esta
# em testes_S4, e por isso colabs_S3 tambem e varrida.
BASES = [
    ".", "..", os.path.join("..", ".."), "/content",
    "notebooks",
    os.path.join("notebooks", "colabs_S4", "testes_extras", "testes_S4"),
    os.path.join("..", "testes_S4"),
    os.path.join("notebooks", "colabs_S3"),
    os.path.join("..", "..", "..", "colabs_S3"),
    os.path.join("notebooks", "colabs_S3", "preparacao_testes"),
    os.path.join("..", "..", "..", "colabs_S3", "preparacao_testes"),
]


def resolve(arquivo):
    """Primeiro caminho existente para o arquivo, entre as bases candidatas."""
    tentativas = []
    for base in BASES:
        caminho = os.path.join(base, arquivo)
        tentativas.append(caminho)
        if os.path.exists(caminho):
            return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas))


print("Diretorio atual:", os.path.abspath("."))
print("Versoes de tratamento das perdas:", ", ".join(SUFIXOS))
print("Base de referencia das secoes 3 a 10:", SUFIXO)
print("Areas a testar:", len(AREAS))

## 2. Leitura dos datasets e da particao

As colunas de entrada sao reconhecidas pelos prefixos de configuracao; o que sobra sao as
colunas-alvo, uma por componente, com a temperatura em graus Celsius. Os valores ausentes estao nas
colunas-alvo: sao medicoes que nao foram feitas naquele ensaio.

Ensaios do conjunto de teste tem `dobra` vazia na particao, e e assim que eles ficam de fora de tudo
o que se faz aqui.

In [ ]:
PREFIXOS_FEATURES = [
    "blower_direction", "housing", "board_position", "run_case",
    "losses", "deflector_hole", "material_coil_support",
    # Atributos fisicos extraidos do motivo do ensaio, na preparacao.
    # Sem este prefixo declarado, seriam tratados como colunas-alvo.
    "cfg",
]
COLUNAS_AUXILIARES = ["run_idx"]

# Um dicionario de datasets por base. A de referencia fica tambem em `datasets`,
# nome que as secoes seguintes usam.
datasets_por_base = {
    sufixo: {area: pd.read_csv(resolve(f"dataset_{area}{sufixo}.csv")) for area in AREAS}
    for sufixo in SUFIXOS
}
datasets = datasets_por_base[SUFIXO]

particao = pd.read_csv(resolve("particao_treino_teste.csv"))

# Dobra de cada ensaio. Ensaios do teste reservado ficam com None e, por isso,
# sao excluidos de toda a validacao mais adiante.
DOBRA = {int(linha.run_idx): (None if pd.isna(linha.dobra) else int(linha.dobra))
         for linha in particao.itertuples()}
DOBRAS = sorted({d for d in DOBRA.values() if d is not None})

print(f"Bases carregadas: {len(datasets_por_base)}  ({', '.join(SUFIXOS)})")
print(f"Ensaios de treino: {int((particao['conjunto'] == 'treino').sum())}  |  "
      f"teste reservado: {int((particao['conjunto'] == 'teste').sum())}")
print(f"Dobras da validacao cruzada: {len(DOBRAS)}  {DOBRAS}")

for sufixo, base in datasets_por_base.items():
    print()
    print(f"--- {sufixo}  ({NOME_BASE[sufixo]}) ---")
    print("{:<16} {:>8} {:>9} {:>6} {:>7}".format("Area", "Ensaios", "Entradas", "Alvos", "Nulos"))
    print("-" * 50)
    for area, df in base.items():
        entradas = [c for c in df.columns if any(c.startswith(p + "_") for p in PREFIXOS_FEATURES)]
        alvos = [c for c in df.columns if c not in entradas + COLUNAS_AUXILIARES]
        print("{:<16} {:>8} {:>9} {:>6} {:>7}".format(
            area, len(df), len(entradas), len(alvos), int(df[alvos].isna().sum().sum())))

print()
print("A coluna Nulos deve ser zero nas duas bases: e o que o sufixo _snull garante.")


## 3. A formulacao do modelo, reproduzida da Sprint 3

Identica a do teste anterior e a da Secao 19 do `modelagem.ipynb`: um modelo por **area**, sobre o
formato longo em que cada linha e um par de ensaio e componente. Ao vetor de configuracao somam-se o
**patamar termico** (a media do componente) e a **escala** (o desvio-padrao dele), ambos calculados
**somente sobre as linhas de treino de cada dobra** - calcula-los sobre a base inteira levaria
informacao da validacao para dentro do treino.

O formato longo inclui apenas os pares ensaio-componente efetivamente medidos, de modo que nenhuma
imputacao e necessaria e nenhuma metrica sai de valor inventado.

In [ ]:
MIN_TREINO = 8
MIN_PREDICOES = 5


def separar(df, prefixos=PREFIXOS_FEATURES):
    """Devolve (X, cols_alvo). run_idx fica fora de features e de alvos."""
    cols_entrada = [c for c in df.columns if any(c.startswith(p + "_") for p in prefixos)]
    cols_alvo = [c for c in df.columns if c not in cols_entrada + COLUNAS_AUXILIARES]
    return df[cols_entrada].copy(), cols_alvo


def formato_longo(df):
    """Uma linha por par de ensaio e componente medido.

    Copiada da Secao 19 do modelagem.ipynb. Quando a coluna run_idx nao existe
    no dataset, o indice posicional faz o seu papel -- os arquivos preservam a
    ordem dos ensaios da particao.
    """
    X, cols_alvo = separar(df)
    blocos = []
    for col in cols_alvo:
        y = df[col]
        if y.notna().sum() < MIN_PREDICOES:
            continue
        sub = X[y.notna()].copy()
        sub["_run"] = (df["run_idx"][y.notna()].to_numpy()
                       if "run_idx" in df.columns else np.arange(len(df))[y.notna()])
        sub["_componente"] = col
        sub["_alvo"] = y[y.notna()].to_numpy()
        blocos.append(sub)
    longo = pd.concat(blocos, ignore_index=True)
    longo["_dobra"] = longo["_run"].map(DOBRA)
    # Linhas sem dobra sao as do conjunto de teste reservado, que fica de fora.
    return longo[longo["_dobra"].notna()].reset_index(drop=True)


def atributos_do_componente(longo, treino):
    """Patamar e escala de cada componente, medidos so no treino da dobra."""
    g = longo[treino].groupby("_componente")["_alvo"]
    media = g.mean()
    desvio = g.std().fillna(1.0).replace(0, 1.0)
    geral = float(longo.loc[treino, "_alvo"].mean())
    return media, desvio, geral


def matriz(longo, mascara, entradas, media, desvio, geral):
    """Vetor de configuracao acrescido do patamar e da escala do componente."""
    comp = longo.loc[mascara, "_componente"]
    return np.hstack([
        longo.loc[mascara, entradas].to_numpy(),
        comp.map(media).fillna(geral).to_numpy().reshape(-1, 1),
        comp.map(desvio).fillna(1.0).to_numpy().reshape(-1, 1),
    ])


print("{:<16} {:>10} {:>12} {:>12}".format("Area", "Linhas", "Componentes", "Colunas"))
print("-" * 54)
for area, df in datasets.items():
    longo = formato_longo(df)
    entradas = [c for c in longo.columns if not c.startswith("_")]
    print("{:<16} {:>10} {:>12} {:>12}".format(
        area, len(longo), longo["_componente"].nunique(), len(entradas) + 2))
print()
print("A coluna 'Colunas' ja inclui os dois atributos do componente.")

## 4. Metricas

**MAE** e o erro absoluto medio em graus, o numero principal. **RMSE** pune erros grandes mais do que
o MAE, e por isso a distancia entre os dois indica se os erros estao concentrados em poucos casos.
**R2** e a fracao da variacao do alvo que o modelo explica; negativo significa pior do que chutar a
media historica do componente.

As tres sao calculadas por componente e depois promediadas dentro da area (macro-media), o que da o
mesmo peso a cada componente.

In [ ]:
def metricas_por_componente(real, previsto):
    """MAE, RMSE e R2 de um componente, com o alvo em graus Celsius."""
    real = np.asarray(real, dtype=float).ravel()
    previsto = np.asarray(previsto, dtype=float).ravel()
    if real.shape != previsto.shape:
        raise ValueError(f"dimensoes diferentes: {real.shape} e {previsto.shape}")
    if real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(real).any() or np.isnan(previsto).any():
        raise ValueError("ha valores ausentes; a mascara deve ser aplicada antes")

    erro = previsto - real
    mae = float(np.mean(np.abs(erro)))
    # A raiz e explicita: o parametro squared do mean_squared_error foi removido
    # do scikit-learn na versao 1.9.
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(real).size < 2:
        r2 = float("nan")   # indefinido quando o alvo assume um unico valor
    else:
        r2 = 1.0 - float(np.sum(erro ** 2)) / float(np.sum((real - real.mean()) ** 2))

    return mae, rmse, r2


def macro_da_area(g, previsto):
    """Media das metricas sobre os componentes da area."""
    previsto = pd.Series(np.asarray(previsto, dtype=float), index=g.index)
    maes, rmses, r2s = [], [], []
    for _, bloco in g.groupby("componente"):
        mae, rmse, r2 = metricas_por_componente(bloco["real"], previsto.loc[bloco.index])
        maes.append(mae)
        rmses.append(rmse)
        if not np.isnan(r2):
            r2s.append(r2)
    return {
        "MAE": float(np.mean(maes)),
        "RMSE": float(np.mean(rmses)),
        "R2": float(np.mean(r2s)) if r2s else float("nan"),
        "componentes": len(maes),
        "n": int(len(g)),
    }


print("Metricas definidas: metricas_por_componente, macro_da_area")

## 5. Treino dos tres algoritmos

Validacao **leave-one-group-out** sobre a coluna `dobra`: a cada rodada, uma configuracao inteira sai
do treino e serve de validacao. Os tres algoritmos sao treinados sobre exatamente a mesma matriz, e
as previsoes dos tres ficam guardadas lado a lado - e isso que permite testar quantas combinacoes de
pesos se queira sem retreinar nada.

### Uma simplificacao que a troca permite

O teste anterior precisava padronizar as entradas so para o Ridge, porque modelo linear com
penalizacao e sensivel a escala das colunas. **Com os tres algoritmos sendo arvores, esse cuidado
deixa de ser necessario**: arvores decidem por cortes em cada coluna separadamente, e multiplicar uma
coluna por uma constante nao muda corte algum. Os tres recebem exatamente as mesmas entradas, sem
escalonamento, e a comparacao entre eles fica mais limpa do que era.

### Hiperparametros

Todos vindos dos notebooks do projeto, para que a comparacao seja com os modelos ja medidos:

- **Extra Trees** - 300 arvores, `random_state=42`, como no `treino_extratrees_7areas.ipynb`.
- **Random Forest** - 300 arvores, `random_state=42`, como na Secao 19 do `modelagem.ipynb`.
- **Gradient Boosting** - 300 arvores e taxa de aprendizado de 0,05, idem.

Esta e a celula demorada do notebook: sete areas vezes dezessete dobras vezes tres algoritmos.

In [ ]:
from sklearn.ensemble import (ExtraTreesRegressor, RandomForestRegressor,
                              GradientBoostingRegressor)

ET_PARAMS = dict(n_estimators=300, random_state=42, n_jobs=-1)
RF_PARAMS = dict(n_estimators=300, random_state=42, n_jobs=-1)
GB_PARAMS = dict(n_estimators=300, learning_rate=0.05, random_state=42)

MODELOS = ["et", "rf", "gb"]
NOME_LONGO = {"et": "Extra Trees", "rf": "Random Forest", "gb": "Gradient Boosting"}


def treinar(base):
    """Previsoes de validacao dos tres algoritmos, para um dicionario de datasets.

    A funcao e a mesma rotina de antes, so que parametrizada pela base: assim as
    duas versoes de tratamento das perdas passam exatamente pelo mesmo caminho,
    com as mesmas dobras e as mesmas sementes.
    """
    blocos_previsao = []
    for area, df in base.items():
        longo = formato_longo(df)
        entradas = [c for c in longo.columns if not c.startswith("_")]

        for d in DOBRAS:
            treino = (longo["_dobra"] != d).to_numpy()
            validacao = (longo["_dobra"] == d).to_numpy()
            if treino.sum() < MIN_TREINO or not validacao.any():
                continue

            # Patamar e escala: so o treino desta dobra e consultado.
            media, desvio, geral = atributos_do_componente(longo, treino)
            X_tr = matriz(longo, treino, entradas, media, desvio, geral)
            X_val = matriz(longo, validacao, entradas, media, desvio, geral)
            y_tr = longo.loc[treino, "_alvo"]

            # Tres conjuntos de arvores, mesmas entradas, sem escalonamento.
            bloco = longo.loc[validacao, ["_componente", "_run", "_alvo"]].copy()
            bloco.columns = ["componente", "ensaio", "real"]
            bloco["dataset"] = area
            bloco["dobra"] = d
            bloco["et"] = ExtraTreesRegressor(**ET_PARAMS).fit(X_tr, y_tr).predict(X_val)
            bloco["rf"] = RandomForestRegressor(**RF_PARAMS).fit(X_tr, y_tr).predict(X_val)
            bloco["gb"] = GradientBoostingRegressor(**GB_PARAMS).fit(X_tr, y_tr).predict(X_val)
            blocos_previsao.append(bloco)

    return pd.concat(blocos_previsao, ignore_index=True)


# As duas bases sao treinadas aqui. A de referencia fica tambem em `predicoes`.
predicoes_por_base = {sufixo: treinar(base) for sufixo, base in datasets_por_base.items()}
predicoes = predicoes_por_base[SUFIXO]

for sufixo, p in predicoes_por_base.items():
    print(f"--- {sufixo}  ({NOME_BASE[sufixo]}) ---")
    print(f"  Previsoes de validacao: {len(p)}  |  componentes: {p['componente'].nunique()}"
          f"  |  ensaios: {p['ensaio'].nunique()}")
    for m in MODELOS:
        print(f"  {NOME_LONGO[m]:<18} {np.abs(p[m] - p['real']).mean():.3f} C")
    print()


### Quanto os tres discordam entre si

Uma mescla so tem do que tirar proveito se os modelos discordarem. A celula abaixo mede isso
diretamente: a diferenca absoluta media entre as previsoes de cada par, em graus.

E aqui que se ve se a troca do Ridge pelo Extra Trees faz sentido pelo motivo alegado. Se o Extra
Trees discordar do Random Forest tanto quanto o Gradient Boosting discorda, ele traz informacao nova
para a media. Se discordar muito pouco, ele e quase uma copia e a mescla nao tem o que aproveitar.

In [ ]:
print("Discordancia media entre pares de modelos, em graus:")
print()
print("{:<36} {:>10}".format("par", "|dif| media"))
print("-" * 48)
for i, a in enumerate(MODELOS):
    for b in MODELOS[i + 1:]:
        dif = float(np.abs(predicoes[a] - predicoes[b]).mean())
        print("{:<36} {:>10.3f}".format(f"{NOME_LONGO[a]} x {NOME_LONGO[b]}", dif))
print()
print("Para referencia, o erro dos proprios modelos esta entre 6 e 8 graus.")

### Conferencia do isolamento do conjunto de teste

Antes de medir qualquer coisa, a confirmacao por numero de que nenhum ensaio reservado entrou na
validacao.

In [ ]:
ensaios_teste = {int(l.run_idx) for l in particao.itertuples() if l.conjunto == "teste"}

print(f"Ensaios do teste reservado: {len(ensaios_teste)}")
falhou = False
for sufixo, p in predicoes_por_base.items():
    invasores = sorted(set(p["ensaio"].astype(int)) & ensaios_teste)
    falhou = falhou or bool(invasores)
    print(f"  {sufixo:<16} reservados que apareceram na validacao: {len(invasores)}"
          + ("" if not invasores else f"  -- {invasores}"))
print("Verificacao:", "FALHOU" if falhou else "PASSOU")


## 6. A grade de pesos

Cada combinacao e uma tripla `(peso_et, peso_rf, peso_gb)` que soma 1. A grade e maior que a do teste
anterior, por duas razoes.

A primeira e que agora os tres algoritmos sao competitivos, e faz sentido testar todos os pares - no
teste anterior, qualquer combinacao com Ridge estava condenada de antemao.

A segunda e que o teste anterior deixou um buraco: em seis das sete areas o melhor resultado caiu na
borda `rf = 1,00`, e o vizinho mais proximo testado era `rf = 0,50`. Nao havia nada entre os dois.
As combinacoes `0,70 / 0,30` desta grade fecham esse vao, e sao candidatas legitimas a vencer.

A grade cobre:

- **Os tres modelos puros**, que sao a referencia sem a qual nao se sabe se a mescla ganhou algo.
- **Os tres pares**, cada um em proporcoes diferentes: Extra Trees com Random Forest, Random Forest
  com Gradient Boosting, Extra Trees com Gradient Boosting.
- **Combinacoes dos tres**, da media simples a repartições que favorecem um deles.

In [ ]:
GRADE_PESOS = [
    # modelos puros
    (1.00, 0.00, 0.00),
    (0.00, 1.00, 0.00),
    (0.00, 0.00, 1.00),
    # pares Extra Trees + Random Forest
    (0.50, 0.50, 0.00),
    (0.70, 0.30, 0.00),
    (0.30, 0.70, 0.00),
    # pares Random Forest + Gradient Boosting
    (0.00, 0.50, 0.50),
    (0.00, 0.70, 0.30),
    (0.00, 0.30, 0.70),
    # par Extra Trees + Gradient Boosting
    (0.50, 0.00, 0.50),
    # os tres juntos
    (0.33, 0.33, 0.34),
    (0.20, 0.40, 0.40),
    (0.10, 0.30, 0.60),
    (0.40, 0.40, 0.20),
    (0.25, 0.50, 0.25),
    (0.45, 0.45, 0.10),
]

NOMES_PUROS = {
    (1.00, 0.00, 0.00): "Extra Trees puro",
    (0.00, 1.00, 0.00): "Random Forest puro",
    (0.00, 0.00, 1.00): "Gradient Boosting puro",
}

for pesos in GRADE_PESOS:
    if abs(sum(pesos) - 1.0) > 1e-9:
        raise ValueError(f"os pesos {pesos} nao somam 1")
if len(set(GRADE_PESOS)) != len(GRADE_PESOS):
    raise ValueError("ha combinacao de pesos repetida na grade")


def rotulo(pesos):
    base = "et {:.2f} | rf {:.2f} | gb {:.2f}".format(*pesos)
    return base + ("   <- " + NOMES_PUROS[pesos] if pesos in NOMES_PUROS else "")


print(f"Combinacoes de pesos a testar: {len(GRADE_PESOS)}")
for pesos in GRADE_PESOS:
    print(" ", rotulo(pesos))

## 7. Resultado por area

A tabela de cada area lista todas as combinacoes, ordenadas do menor para o maior MAE. Os modelos
puros vem marcados, para que se veja de imediato se alguma mescla os superou - e por quanto.

In [ ]:
def tabela_resultados(predicoes):
    """Uma linha por par de area e combinacao de pesos, com as metricas macro."""
    linhas = []
    for area, g in predicoes.groupby("dataset"):
        P = g[MODELOS].to_numpy()
        for pesos in GRADE_PESOS:
            linhas.append({
                "area": area,
                "pesos": rotulo(pesos),
                "w_et": pesos[0], "w_rf": pesos[1], "w_gb": pesos[2],
                "puro": NOMES_PUROS.get(pesos, ""),
                "usa_et": pesos[0] > 0,
                **macro_da_area(g, P @ np.array(pesos, dtype=float)),
            })
    return pd.DataFrame(linhas)


resultados_por_base = {sufixo: tabela_resultados(p)
                       for sufixo, p in predicoes_por_base.items()}
resultados = resultados_por_base[SUFIXO]

print(f"Linhas de resultado por base: {len(resultados)}  "
      f"({len(AREAS)} areas x {len(GRADE_PESOS)} combinacoes de pesos)")
print(f"Bases avaliadas: {', '.join(resultados_por_base)}")


In [ ]:
for area in AREAS:
    bloco = resultados[resultados["area"] == area].sort_values("MAE")
    primeira = bloco.iloc[0]
    print("=" * 76)
    print(f"{area}   |   {int(primeira['componentes'])} componentes   |   "
          f"{int(primeira['n'])} previsoes de validacao")
    print("=" * 76)
    print(bloco[["pesos", "MAE", "RMSE", "R2"]].round(3).to_string(index=False))
    print()

## 8. Tabela resumo - a mescla vale a pena?

A comparacao e entre a melhor combinacao da area e o melhor modelo puro da mesma area. O ganho e a
diferenca entre os dois MAEs, em graus: positivo significa que a mescla errou menos. Quando
`mescla_venceu` e falso, o melhor resultado da area foi um modelo puro.

In [ ]:
resumo = []
for area in AREAS:
    bloco = resultados[resultados["area"] == area]
    melhor = bloco.loc[bloco["MAE"].idxmin()]
    puros = bloco[bloco["puro"] != ""]
    melhor_puro = puros.loc[puros["MAE"].idxmin()]
    ganho = float(melhor_puro["MAE"] - melhor["MAE"])
    resumo.append({
        "area": area,
        "melhor_peso": melhor["pesos"].split("   <-")[0],
        "MAE_melhor": float(melhor["MAE"]),
        "melhor_puro": melhor_puro["puro"],
        "MAE_puro": float(melhor_puro["MAE"]),
        "ganho_C": ganho,
        "ganho_%": 100.0 * ganho / float(melhor_puro["MAE"]),
        "mescla_venceu": melhor["puro"] == "",
    })

df_resumo = pd.DataFrame(resumo)
print(df_resumo.round(3).to_string(index=False))
print()
print(f"Areas em que a mescla superou todos os modelos puros: "
      f"{int(df_resumo['mescla_venceu'].sum())} de {len(AREAS)}")
print(f"Ganho medio sobre o melhor puro: {df_resumo['ganho_C'].mean():+.3f} C "
      f"({df_resumo['ganho_%'].mean():+.2f}%)")

### Os tres algoritmos sozinhos

Antes de avaliar a mescla, vale ver como cada um se sai por conta propria. E o que diz se o Extra
Trees e um competidor de verdade nesta base ou apenas um passageiro.

In [ ]:
print("{:<16} {:>12} {:>14} {:>18} {:>10}".format(
    "area", "Extra Trees", "Random Forest", "Gradient Boosting", "melhor"))
print("-" * 76)

vitorias = {m: 0 for m in MODELOS}
for area in AREAS:
    bloco = resultados[(resultados["area"] == area) & (resultados["puro"] != "")]
    valores = {}
    for m in MODELOS:
        chave = f"{NOME_LONGO[m]} puro"
        valores[m] = float(bloco[bloco["puro"] == chave]["MAE"].iloc[0])
    vencedor = min(valores, key=valores.get)
    vitorias[vencedor] += 1
    print("{:<16} {:>12.3f} {:>14.3f} {:>18.3f} {:>10}".format(
        area, valores["et"], valores["rf"], valores["gb"], vencedor.upper()))

print()
for m in MODELOS:
    print(f"  {NOME_LONGO[m]:<18} e o melhor modelo puro em {vitorias[m]} de {len(AREAS)} areas")

## 9. O Extra Trees ganha o seu lugar?

Esta e a secao que responde a pergunta do notebook, e ela nao se responde olhando a tabela de melhor
peso. O motivo: uma combinacao pode vencer **apesar** do Extra Trees, e nao por causa dele. Se a
melhor da area for `et 0,00 | rf 0,50 | gb 0,50`, o Extra Trees nao participou, e o resultado e o
mesmo que o teste anterior ja tinha.

O teste correto e comparar, dentro de cada area, **a melhor combinacao que usa Extra Trees contra a
melhor que nao usa**. A diferenca entre as duas e exatamente o que ele acrescenta - nem mais, nem
menos.

Sobre o limiar: diferencas muito pequenas nao devem ser lidas como diferenca de desempenho, porque os
pesos foram escolhidos sobre o mesmo conjunto que os avalia. Adoto **0,10 grau** como piso do que
conta como ganho apreciavel, valor abaixo do qual a diferenca e menor que o ruido experimental de
qualquer area da base. Ele e uma convencao deste notebook, declarada para que a leitura nao dependa
de impressao.

In [ ]:
GANHO_MINIMO = 0.10   # graus; abaixo disso a diferenca nao e lida como ganho

comparacao = []
for area in AREAS:
    bloco = resultados[resultados["area"] == area]
    com = bloco[bloco["usa_et"]]
    sem = bloco[~bloco["usa_et"]]
    melhor_com = com.loc[com["MAE"].idxmin()]
    melhor_sem = sem.loc[sem["MAE"].idxmin()]
    ganho = float(melhor_sem["MAE"] - melhor_com["MAE"])
    comparacao.append({
        "area": area,
        "melhor_com_ET": melhor_com["pesos"].split("   <-")[0],
        "MAE_com_ET": float(melhor_com["MAE"]),
        "melhor_sem_ET": melhor_sem["pesos"].split("   <-")[0],
        "MAE_sem_ET": float(melhor_sem["MAE"]),
        "ganho_do_ET": ganho,
        "veredito": ("ACRESCENTA" if ganho >= GANHO_MINIMO
                     else ("nao muda" if ganho > -GANHO_MINIMO else "ATRAPALHA")),
    })

df_et = pd.DataFrame(comparacao)
print(df_et.round(3).to_string(index=False))
print()

acrescenta = int((df_et["veredito"] == "ACRESCENTA").sum())
neutro = int((df_et["veredito"] == "nao muda").sum())
atrapalha = int((df_et["veredito"] == "ATRAPALHA").sum())
print(f"Acrescenta (ganho >= {GANHO_MINIMO} C): {acrescenta} de {len(AREAS)} areas")
print(f"Nao muda nada:                       {neutro} de {len(AREAS)} areas")
print(f"Atrapalha:                           {atrapalha} de {len(AREAS)} areas")
print()

if acrescenta == 0:
    print("CONCLUSAO: o Extra Trees NAO ganha o seu lugar na mescla. Em nenhuma area ele melhora")
    print("o resultado alem do limiar adotado, e o teste com Ridge ja dizia tudo o que ha a dizer.")
elif acrescenta == len(AREAS):
    print("CONCLUSAO: o Extra Trees ganha o seu lugar em todas as areas.")
else:
    ganhadoras = ", ".join(df_et.loc[df_et["veredito"] == "ACRESCENTA", "area"])
    print(f"CONCLUSAO: o Extra Trees so ganha o seu lugar em {acrescenta} das {len(AREAS)} areas "
          f"({ganhadoras}).")
    print("Nas demais ele nao acrescenta o suficiente para justificar um terceiro modelo em")
    print("producao, e a decisao deve ser tomada area por area, e nao como regra geral.")

### O custo do que se ganha

Um terceiro algoritmo na mescla nao e de graca: e mais um modelo para treinar, versionar, monitorar
e explicar a quem recebe a previsao. A celula abaixo poe o ganho em perspectiva, comparando-o com o
proprio erro da area e com o ganho que a formulacao por area trouxe na Sprint 3 (0,379 grau sobre a
formulacao por componente, com significancia estatistica medida).

In [ ]:
print("{:<16} {:>12} {:>14} {:>16}".format(
    "area", "ganho do ET", "erro da area", "ganho / erro"))
print("-" * 62)
for _, linha in df_et.iterrows():
    base = linha["MAE_sem_ET"]
    print("{:<16} {:>+12.3f} {:>14.3f} {:>15.1f}%".format(
        linha["area"], linha["ganho_do_ET"], base, 100 * linha["ganho_do_ET"] / base))

print()
print(f"Ganho medio do Extra Trees sobre as sete areas: "
      f"{df_et['ganho_do_ET'].mean():+.3f} C")
print("Para comparacao, a mudanca para a formulacao por area, na Sprint 3, valeu 0,379 C.")

## 10. Verificacao de sanidade - o Random Forest puro bate com a Sprint 3?

O mesmo controle do teste anterior, e pela mesma razao: se a formulacao tiver sido reproduzida
errado, o notebook roda ate o fim e produz tabelas de aparencia normal, so que sobre um modelo que
nao e o de producao. Como `(0, 1, 0)` e o Random Forest puro, ou seja, o modelo entregue na Sprint 3,
o MAE dessa linha tem que reproduzir o que a entrega registrou.

A tolerancia e uma faixa porque a referencia foi medida sobre a base `_perfil` e aqui a base e a
`_13cat`, regerada depois. O alerta exige que a diferenca passe de 1,5 grau **e** de 30%, para nao
acusar por variacao trivial uma area cujo erro ja e pequeno.

In [ ]:
# MAE por area do Random Forest sob a formulacao por area, registrado na
# documentacao da Sprint 3 (base _perfil, mesmas dobras).
REFERENCIA_SPRINT3 = {
    "oven_hmi": 2.42,
    "ct_knobs": 3.16,
    "ipc_cooling": 4.04,
    "main_vent_sys": 4.68,
    "mainback": 5.68,
    "ipc_comp": 7.30,
    "coil_temps": 11.71,
}

TOL_ABS, TOL_REL = 1.5, 0.30

print("{:<16} {:>10} {:>12} {:>12} {:>10}  {}".format(
    "area", "RF aqui", "Sprint 3", "diferenca", "dif %", "situacao"))
print("-" * 76)

alertas = []
for area in AREAS:
    bloco = resultados[(resultados["area"] == area)
                       & (resultados["puro"] == "Random Forest puro")]
    aqui = float(bloco["MAE"].iloc[0])
    ref = REFERENCIA_SPRINT3[area]
    dif = aqui - ref
    fora = abs(dif) > TOL_ABS and abs(dif / ref) > TOL_REL
    if fora:
        alertas.append(area)
    print("{:<16} {:>10.2f} {:>12.2f} {:>+12.2f} {:>+9.1f}%  {}".format(
        area, aqui, ref, dif, 100 * dif / ref, "DIVERGE" if fora else "compativel"))

print()
if alertas:
    print("ATENCAO: o MAE do Random Forest puro diverge da referencia da Sprint 3 nas areas:")
    print("  " + ", ".join(alertas))
    print("A divergencia pode vir da diferenca entre as bases _perfil e _13cat, ou da regeracao")
    print("dos arquivos em testes_S4 -- mas tambem pode indicar que a formulacao foi reproduzida")
    print("de forma errada. Enquanto isso nao for esclarecido, os resultados acima NAO devem ser")
    print("lidos como comparaveis com a entrega da Sprint 3.")
else:
    print("VERIFICACAO PASSOU: o Random Forest puro reproduz o desempenho da entrega da Sprint 3")
    print("em todas as areas, dentro da tolerancia. A formulacao esta reproduzida corretamente e")
    print("a comparacao entre as combinacoes de pesos e valida.")

## 11. As perdas agrupadas por perfil treinam melhor ou pior?

Ate aqui tudo correu sobre a base `_13cat_snull`. Esta secao repete a leitura sobre a
`_perfil_snull` e poe as duas lado a lado.

A diferenca entre elas e apenas a **codificacao das perdas**. Na versao de 13 categorias, cada
combinacao de perdas declarada no motivo do ensaio vira uma coluna propria - sao 13 colunas, varias
delas acesas em pouquissimos ensaios. Na versao por perfil, essas 13 combinacoes sao agrupadas nos
tres perfis que as originam (`miro`, `simone` e `no losses`), e restam tres colunas. Todo o resto -
ensaios, componentes, dobras, algoritmos, sementes - e identico.

O que se espera de um agrupamento assim e a troca de sempre: perde-se detalhe e ganha-se densidade.
Uma coluna acesa em tres ensaios quase nao tem de onde tirar um padrao; a mesma informacao diluida em
uma coluna acesa em vinte ensaios tem. Se o detalhe descartado nao carregava sinal, o agrupamento
ajuda. Se carregava, atrapalha. Qual dos dois acontece e o que as tabelas abaixo medem, e a resposta
nao precisa ser a mesma em todas as areas.

### O que a comparacao mede, e o que ela nao mede

Os numeros abaixo comparam o **erro da validacao cruzada** das duas bases. Duas ressalvas valem o
registro, porque a tabela sozinha nao as mostra:

1. **O sufixo `_snull` imputa valores, e isso desloca o nivel do erro.** As colunas-alvo com
   medicao ausente foram preenchidas na preparacao, e essas linhas imputadas entram tanto no treino
   quanto na validacao. Parte do erro medido aqui e, portanto, erro contra um valor que nao foi
   medido em bancada - e nao e pouco. No `coil_temps`, a area mais afetada, o Random Forest puro
   erra **11,93 graus** sobre a base `_perfil` com os nulos ainda ausentes (344 linhas de validacao)
   e **14,87** sobre a `_perfil_snull` (368 linhas): as 24 linhas imputadas que entram respondem
   sozinhas por quase tres graus de erro medio. O mesmo vale para a `_13cat`, que vai de 11,37 para
   14,57.

   A consequencia pratica e dupla. A **comparacao** entre as duas bases continua valida, porque as
   duas recebem exatamente a mesma imputacao nas mesmas linhas. Mas o **nivel** do erro relatado
   aqui nao e comparavel com o da entrega da Sprint 3, nem com o de qualquer teste feito sobre
   medicoes apenas - e nao deve ser citado como o erro do modelo.
2. **A melhor combinacao de pesos foi escolhida sobre a mesma validacao que a avalia**, nas duas
   bases. A tabela da melhor mescla herda esse otimismo. E por isso que a comparacao principal, logo
   abaixo, e feita sobre os **modelos puros**, que nao passam por escolha alguma.

### Os algoritmos puros, nas duas bases

In [ ]:
# MAE agregado de cada algoritmo puro, nas duas bases. Esta e a comparacao mais
# limpa que ha aqui: nenhum peso foi escolhido, nada foi otimizado.
print("MAE agregado sobre todas as previsoes de validacao, por base e algoritmo:")
print()
print("{:<20} {:>14} {:>16} {:>12}".format("algoritmo", "13 categorias", "perfil de perdas", "diferenca"))
print("-" * 66)

for m in MODELOS:
    a = float(np.abs(predicoes_por_base["_13cat_snull"][m]
                     - predicoes_por_base["_13cat_snull"]["real"]).mean())
    b = float(np.abs(predicoes_por_base["_perfil_snull"][m]
                     - predicoes_por_base["_perfil_snull"]["real"]).mean())
    print("{:<20} {:>14.3f} {:>16.3f} {:>+12.3f}".format(NOME_LONGO[m], a, b, b - a))

print()
print("A coluna 'diferenca' e perfil menos 13 categorias: negativo significa que o")
print("agrupamento por perfil errou MENOS, ou seja, treinou melhor.")


### Area por area

In [ ]:
# Por area, comparando duas coisas: o melhor modelo puro (sem escolha de pesos) e
# a melhor mescla (com escolha, e por isso otimista nas duas bases).
def melhores_da_area(res, area):
    bloco = res[res["area"] == area]
    puros = bloco[bloco["puro"] != ""]
    melhor_puro = puros.loc[puros["MAE"].idxmin()]
    melhor = bloco.loc[bloco["MAE"].idxmin()]
    return float(melhor_puro["MAE"]), float(melhor["MAE"])


LIMIAR = 0.10   # graus; o mesmo limiar de relevancia usado nas secoes anteriores

comparacao_bases = []
for area in AREAS:
    puro_13, mescla_13 = melhores_da_area(resultados_por_base["_13cat_snull"], area)
    puro_pf, mescla_pf = melhores_da_area(resultados_por_base["_perfil_snull"], area)
    dif = puro_pf - puro_13
    comparacao_bases.append({
        "area": area,
        "puro_13cat": puro_13,
        "puro_perfil": puro_pf,
        "dif_puro": dif,
        "dif_%": 100.0 * dif / puro_13,
        "mescla_13cat": mescla_13,
        "mescla_perfil": mescla_pf,
        "veredito": ("perfil MELHOR" if dif <= -LIMIAR
                     else ("perfil pior" if dif >= LIMIAR else "empate")),
    })

df_bases = pd.DataFrame(comparacao_bases)
print(df_bases.round(3).to_string(index=False))
print()

melhor = int((df_bases["veredito"] == "perfil MELHOR").sum())
pior = int((df_bases["veredito"] == "perfil pior").sum())
empate = int((df_bases["veredito"] == "empate").sum())
print(f"Areas em que o perfil treina melhor: {melhor}   pior: {pior}   empate: {empate}"
      f"   (limiar de {LIMIAR:.2f} C)")
print(f"Diferenca media sobre as sete areas: {df_bases['dif_puro'].mean():+.3f} C"
      f"  ({df_bases['dif_%'].mean():+.1f}%)")
print()
print("Lembrando: a media sobre as areas pesa igual uma area de erro grande e uma de")
print("erro pequeno. Por isso o veredito por area importa mais do que a media.")


### O veredito

In [ ]:
# O veredito, escrito por extenso a partir dos numeros acima -- nao a mao.
media_dif = float(df_bases["dif_puro"].mean())
piores = df_bases[df_bases["veredito"] == "perfil pior"]["area"].tolist()
melhores = df_bases[df_bases["veredito"] == "perfil MELHOR"]["area"].tolist()

print("=" * 76)
print("VEREDITO: o treinamento com as perdas agrupadas por perfil")
print("=" * 76)
print()

# Duas leituras do mesmo dado, que podem discordar -- e quando discordam, isso
# e a informacao principal, nao um detalhe a esconder.
if melhor > pior:
    contagem = f"vence em {melhor} das {len(AREAS)} areas e perde em {pior}"
    lado_contagem = "melhor"
elif pior > melhor:
    contagem = f"perde em {pior} das {len(AREAS)} areas e vence em {melhor}"
    lado_contagem = "pior"
else:
    contagem = f"vence em {melhor} areas e perde em {pior}"
    lado_contagem = "empate"

lado_media = ("melhor" if media_dif <= -0.01
              else ("pior" if media_dif >= 0.01 else "empate"))

print(f"Por contagem de areas:   {contagem}.")
print(f"Por diferenca media:     {media_dif:+.3f} C no melhor modelo puro"
      f"  ({df_bases['dif_%'].mean():+.1f}% em media relativa).")
print()

FRASE = {"melhor": "favorece o perfil",
         "pior": "favorece as 13 categorias",
         "empate": "nao separa as duas"}

if lado_contagem == lado_media:
    if lado_contagem == "empate":
        print("As duas leituras concordam: nao ha diferenca que se sustente entre as bases.")
    else:
        print(f"As duas leituras concordam: o agrupamento por perfil treina "
              f"{lado_contagem.upper()}.")
else:
    print("ATENCAO: as duas leituras NAO apontam para o mesmo lado, e por isso o")
    print("resultado nao deve ser resumido num veredito unico.")
    print(f"  - a contagem de areas {FRASE[lado_contagem]};")
    print(f"  - a diferenca media {FRASE[lado_media]}.")
    print("Quando isso acontece, o motivo esta no tamanho dos ganhos e das perdas, e")
    print("nao na quantidade delas: um punhado de areas que perde por muito desloca a")
    print("media inteira. Os dois numeros abaixo mostram exatamente isso.")

print()
if melhores:
    ganho = df_bases[df_bases["area"].isin(melhores)]["dif_puro"]
    print(f"Melhor com perfil ({len(melhores)}): " + ", ".join(melhores))
    print(f"   ganho medio onde ganha: {-ganho.mean():.3f} C")
if piores:
    perda = df_bases[df_bases["area"].isin(piores)]["dif_puro"]
    print(f"Pior com perfil ({len(piores)}):   " + ", ".join(piores))
    print(f"   perda media onde perde: {perda.mean():.3f} C")

print()
print("A leitura util nao e um sim ou nao global. O agrupamento por perfil troca")
print("detalhe por densidade: cada coluna de perdas passa a ser vista em mais")
print("ensaios, e em troca some a distincao entre combinacoes. Onde as 13")
print("categorias tinham colunas acesas em pouquissimos ensaios, essa troca")
print("compensa; onde a distincao entre combinacoes carregava sinal de verdade,")
print("ela apaga esse sinal. A decisao, portanto, e por area -- e a tabela acima")
print("diz quais areas ficam de cada lado.")


## 12. Como ler este resultado

**O peso foi escolhido sobre a mesma validacao que o avalia.** Dezesseis combinacoes foram comparadas
em cada area e a melhor foi reportada. Escolher a melhor entre dezesseis sobre um mesmo conjunto
captura parte do ruido junto com o sinal, e o ganho medido e por isso uma estimativa otimista. Note
que essa grade e maior que a do teste com Ridge, o que **aumenta** o otimismo, e nao diminui: quanto
mais candidatos, maior a chance de um deles parecer bom por acaso. Essa e a razao do limiar de 0,10
grau adotado na Secao 9.

**O conjunto de teste reservado continua intacto.** Nenhum ensaio dele entrou em etapa alguma - o que
significa tambem que nada aqui foi confirmado em dado independente.

**Erro medio nao e a unica medida que importa.** A Sprint 3 registrou que o algoritmo de menor erro
nao e o que deixa passar menos excessos sobre o limite de seguranca: o Gradient Boosting errava mais
em media e deixava passar menos casos perigosos, 26 contra 36 na margem de 8 graus. Uma mescla que
reduz o MAE pode piorar a decisao contra o limite. Verificar isso exige a tabela `limites_ativos.csv`
e fica fora do escopo deste notebook, que mede erro, e nao seguranca.

### Resultado obtido na execucao de referencia

Os numeros abaixo sao de uma execucao completa deste notebook, feita em 18 de setembro de 2026,
sobre a base de referencia `_13cat_snull`, e servem para conferir se uma nova execucao reproduz o
mesmo quadro.

**Os tres sozinhos.** Agregado sobre as 3.358 previsoes: Random Forest 6,99 graus, Extra Trees 7,28,
Gradient Boosting 7,54. O Extra Trees entra entre os dois - e um competidor de verdade, ao contrario
do Ridge, que ficava em 9,81. Por area, porem, ele so e o melhor modelo puro no **main_vent_sys**
(4,087 contra 4,224 do Random Forest); nas outras seis o Random Forest vence.

**A mescla venceu em 4 das 7 areas**, contra 2 no teste com Ridge: coil_temps (ganho de 0,237 grau
sobre o melhor puro), ipc_cooling (0,142), main_vent_sys (0,032) e ipc_comp (0,030). O tamanho dos
ganhos e muito desigual, e so os de coil_temps e ipc_cooling estao acima do limiar de 0,10 grau.

**Por que o ganho e pequeno: o Extra Trees e quase uma copia do Random Forest.** A medida de
discordancia da Secao 5 mostra o mecanismo. A diferenca media entre as previsoes dos dois e de
apenas **1,65 grau**, ao passo que o Gradient Boosting discorda do Random Forest em 3,20 e do Extra
Trees em 4,01. Ou seja: os dois algoritmos baseados em bagging enxergam a base quase da mesma forma,
e promediar um com o outro cancela pouca coisa. A expectativa de que o sorteio dos cortes tornaria o
Extra Trees suficientemente diferente **nao se confirmou nesta base** - e e essa medida, e nao a
tabela de MAE, que explica por que ele acrescenta tao pouco.

**O Extra Trees so ganha o seu lugar em uma area.** Pela comparacao da Secao 9, entre a melhor
combinacao com ele e a melhor sem ele:

| area | melhor com ET | melhor sem ET | ganho do ET | veredito |
|---|---|---|---|---|
| main_vent_sys | 4,055 | 4,224 | **+0,169** | ACRESCENTA |
| ipc_cooling | 3,746 | 3,813 | +0,067 | nao muda |
| mainback | 5,725 | 5,722 | -0,002 | nao muda |
| ipc_comp | 7,508 | 7,500 | -0,008 | nao muda |
| oven_hmi | 2,559 | 2,517 | -0,042 | nao muda |
| ct_knobs | 3,150 | 3,104 | -0,047 | nao muda |
| coil_temps | 14,538 | 14,336 | -0,203 | ATRAPALHA |

A leitura que esses numeros sustentam: **trocar o Ridge pelo Extra Trees melhora a mescla, mas nao o
suficiente para mudar a recomendacao.** O Extra Trees e de fato um modelo melhor que o Ridge nesta
base, e as combinacoes com ele aparecem bem colocadas em varias areas - mas o ganho liquido sobre o
que ja se conseguia sem ele passa de um decimo de grau em uma unica area, o main_vent_sys. No
coil_temps ele chega a atrapalhar: a melhor combinacao continua sendo a de Random Forest com
Gradient Boosting, e qualquer peso dado ao Extra Trees piora o resultado.

**A comparacao entre as bases (Secao 11).** As duas leituras nao apontam para o mesmo lado. Por
contagem de areas o perfil vence em 3 (ipc_comp, ipc_cooling, oven_hmi) e perde em 2 (coil_temps,
main_vent_sys), com 2 empates; mas por diferenca media do melhor modelo puro ele fica 0,025 grau
**atras**, porque onde perde, perde mais (0,442 em media) do que ganha onde ganha (0,223). No
agregado sobre todas as previsoes, porem, o perfil erra menos com os tres algoritmos: Random Forest
6,891 contra 6,990, Gradient Boosting 7,369 contra 7,543, Extra Trees 7,251 contra 7,284. A decisao
sobre agrupar as perdas por perfil e, portanto, **por area**, e nao uma regra unica.